## 우리금융 추가 실습

## 1. 결측치 처리

In [1]:
import pandas as pd
import numpy as np

c:\Users\User\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\User\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### Titanic 데이터 불러오기

In [2]:
# Titanic 데이터셋 로드
df = pd.read_csv(
    'https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv'
)

print(f'데이터 차원 : {df.shape}')

데이터 차원 : (891, 12)


In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### 결측치 확인

In [4]:
print("\n변수별 결측치 수 확인")
number_null = df.isnull().sum()
print(number_null[number_null != 0])
print("\n결측치 비율 확인")
ratio_null = number_null[number_null != 0] / len(df) * 100
print(ratio_null)
print("\n데이터 타입 확인")
print(df.dtypes[number_null != 0])


변수별 결측치 수 확인
Age         177
Cabin       687
Embarked      2
dtype: int64

결측치 비율 확인
Age         19.865320
Cabin       77.104377
Embarked     0.224467
dtype: float64

데이터 타입 확인
Age         float64
Cabin        object
Embarked     object
dtype: object


### Complete Case Analysis

In [5]:
df_complete = df.dropna()

print('원래 데이터 크기:', df.shape)
print('complete case 크기:', df_complete.shape)
print('삭제된 행 개수:', len(df) - len(df_complete))
print('남은 비율:', len(df_complete) / len(df))

원래 데이터 크기: (891, 12)
complete case 크기: (183, 12)
삭제된 행 개수: 708
남은 비율: 0.2053872053872054


### Cabin 변수 처리 : 객실 구역 정보만 남기기, missing 정보 추가

In [6]:
df_cabin = df.copy()
df_cabin['Cabin'] = df_cabin['Cabin'].str[0] # 객실 구역 정보만 남기기
df_cabin['Cabin'] = df_cabin['Cabin'].fillna('Missing') # missing 정보 추가
df_cabin['Cabin']

0      Missing
1            C
2      Missing
3            C
4      Missing
        ...   
886    Missing
887          B
888    Missing
889          C
890    Missing
Name: Cabin, Length: 891, dtype: object

In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
encoder.fit(df_cabin[['Cabin']])
df_cabin_encoded = encoder.transform(df_cabin[['Cabin']]).toarray().astype(int)
df_cabin_encoded_df = pd.DataFrame(df_cabin_encoded, columns=encoder.get_feature_names_out())
df_cabin_encoded_df
df_cabin = pd.concat([df_cabin[['Cabin']], df_cabin_encoded_df], axis=1)
df_cabin

,Cabin,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_Missing,Cabin_T
0,Missing,0,0,0,0,0,0,0,1,0
1,C,0,0,1,0,0,0,0,0,0
2,Missing,0,0,0,0,0,0,0,1,0
3,C,0,0,1,0,0,0,0,0,0
4,Missing,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...
886,Missing,0,0,0,0,0,0,0,1,0
887,B,0,1,0,0,0,0,0,0,0
888,Missing,0,0,0,0,0,0,0,1,0
889,C,0,0,1,0,0,0,0,0,0


In [8]:
df = pd.merge(df.drop(columns=['Cabin']), df_cabin.drop(columns=['Cabin']), left_index=True, right_index=True)

### Single Imputation

- 평균, 중앙값, 최빈값

In [9]:
# 평균 대체
df_mean = df.copy()
df_mean['Age'] = df_mean['Age'].fillna(df_mean['Age'].mean())

# 중앙값 대체
df_median = df.copy()
df_median['Age'] = df_median['Age'].fillna(df_median['Age'].median())

# 최빈값 대체
df_simple = df.copy()
mode_embarked = df_simple['Embarked'].mode()[0]
df_simple['Embarked'] = df_simple['Embarked'].fillna(mode_embarked)

- 조건부 평균 대체법

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

features = ['Pclass','Sex','SibSp','Parch','Fare','Embarked',
            'Cabin_A', 'Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E','Cabin_F', 
            'Cabin_G', 'Cabin_T', 'Cabin_Missing']
encoder = OneHotEncoder()

df_encoded = encoder.fit_transform(df[features]).toarray()
df_encoded = pd.DataFrame(df_encoded, columns=encoder.get_feature_names_out())
train_age = df_encoded[df['Age'].notnull()]
test_age  = df_encoded[df['Age'].isnull()]

In [11]:
age_model = LinearRegression()
age_model.fit(train_age, df['Age'][df['Age'].notnull()])
pred_age = age_model.predict(test_age)
pred_age = np.clip(pred_age, 0, None)

df_reg = df.copy()
df_reg.loc[df_reg['Age'].isnull(), 'Age'] = pred_age
df_reg['Age'] = df_reg['Age'].astype(int)
print(df_reg['Age'][df['Age'].isnull()])
print("분산 : ",df_reg['Age'][df['Age'].isnull()].var())

5      38
17     33
19     27
26     30
28     19
       ..
859    21
863    26
868    26
878    28
888    29
Name: Age, Length: 177, dtype: int32
분산 :  78.1518361581921


In [12]:
# 확률 보정
train_pred = age_model.predict(train_age)
test_pred  = age_model.predict(test_age)

residuals = df['Age'][df['Age'].notnull()] - train_pred
sigma_hat = residuals.std()

np.random.seed(123)
error = np.random.normal(0, sigma_hat, len(test_pred))

stochastic_age = test_pred + error
stochastic_age = np.clip(stochastic_age, 0, None)

df_reg_prob = df.copy()
df_reg_prob.loc[df_reg_prob['Age'].isnull(), 'Age'] = stochastic_age
df_reg_prob['Age'] = df_reg_prob['Age'].astype(int)
print(df_reg_prob['Age'][df['Age'].isnull()])
print("분산 : ",df_reg_prob['Age'][df['Age'].isnull()].var())

5      27
17     42
19     29
26     16
28     13
       ..
859    14
863    27
868    29
878    42
888    32
Name: Age, Length: 177, dtype: int32
분산 :  188.2242552645095


- Hot Deck Imputation

In [13]:
np.random.seed(123)

def hot_deck_age(row, data):
    if pd.notnull(row['Age']):
        return row['Age']

    same_group = data[
        (data['Sex'] == row['Sex']) &
        (data['Pclass'] == row['Pclass']) &
        (data['Age'].notnull())
    ]['Age']

    if len(same_group) > 0:
        return np.random.choice(same_group)
    return data['Age'].median()

df_hotdeck = df.copy()
df_hotdeck['Age'] = df_hotdeck.apply(lambda row: hot_deck_age(row, df_hotdeck) if pd.isnull(row['Age']) else row['Age'], axis=1)
df_hotdeck['Age'] = df_hotdeck['Age'].astype(int)
df_hotdeck['Age']

0      22
1      38
2      26
3      35
4      35
       ..
886    27
887    19
888    45
889    26
890    32
Name: Age, Length: 891, dtype: int32

### SRMI와 Multiple Imputation

In [14]:
# SRMI와 Multiple Imputation 설명용 4개 변수 simulation 데이터 생성
np.random.seed(0)

n = 10
sim_df = pd.DataFrame({
    'x1': np.random.normal(10, 2, n),
    'x2': np.random.normal(5, 1, n),
    'x3': np.random.normal(3, 1, n),
})

sim_df['y'] = (1.2 * sim_df['x1']
               - 1.5 * sim_df['x2']
               + 2.0 * sim_df['x3']
               + np.random.normal(0, 3, n))

missing_df = sim_df.copy()

# X 변수들(x1, x2, x3)에도 결측치가 불규칙적으로 있도록 결측값 삽입 (각각 10% 결측)
for col in ['x1', 'x2', 'x3']:
    missing_idx = np.random.choice(sim_df.index, size=int(n*0.2), replace=False)
    missing_df.loc[missing_idx, col] = np.nan

missing_df

,x1,x2,x3,y
0,13.528105,5.144044,0.447010,9.876523
1,10.800314,NaN,3.653619,11.720692
2,11.957476,5.761038,3.864436,10.772930
3,14.481786,5.121675,2.257835,8.268912
4,13.735116,5.443863,5.269755,17.812117
5,8.045444,5.333674,1.545634,5.214337
6,NaN,6.494079,3.045759,14.321483
7,9.697286,NaN,NaN,13.677252
8,NaN,5.313068,4.532779,11.686251
9,10.821197,4.145904,NaN,14.798389


In [15]:
# SRMI
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

missing_iter = missing_df.copy()
num_cols = ['x1', 'x2', 'x3', 'y']

imputer = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=20,
    random_state=123
)

missing_iter[num_cols] = imputer.fit_transform(missing_iter[num_cols])
missing_iter

,x1,x2,x3,y
0,13.528105,5.144044,0.447010,9.876523
1,10.800314,5.128735,3.653619,11.720692
2,11.957476,5.761038,3.864436,10.772930
3,14.481786,5.121675,2.257835,8.268912
4,13.735116,5.443863,5.269755,17.812117
5,8.045444,5.333674,1.545634,5.214337
6,15.382940,6.494079,3.045759,14.321483
7,9.697286,4.921054,4.938137,13.677252
8,10.594704,5.313068,4.532779,11.686251
9,10.821197,4.145904,4.967765,14.798389


In [16]:
from sklearn.ensemble import RandomForestRegressor

m=5
imputed_datasets = []

for i in range(m):
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=i),
        max_iter=20,
        random_state=i
    )

    imputed_array = imputer.fit_transform(missing_df[num_cols])

    imputed_df = pd.DataFrame(
        imputed_array,
        columns=missing_df.columns,
        index=missing_df.index
    )

    imputed_datasets.append(imputed_df)

In [17]:
# 다중 대체 결과의 평균값으로 대체
imputed_df = missing_df.copy()
imputed_df = 0

for i in range(m):
    imputed_df += imputed_datasets[i]/m

imputed_df

,x1,x2,x3,y
0,13.528105,5.144044,0.447010,9.876523
1,10.800314,5.084316,3.653619,11.720692
2,11.957476,5.761038,3.864436,10.772930
3,14.481786,5.121675,2.257835,8.268912
4,13.735116,5.443863,5.269755,17.812117
5,8.045444,5.333674,1.545634,5.214337
6,11.421112,6.494079,3.045759,14.321483
7,9.697286,5.205846,3.353679,13.677252
8,12.386077,5.313068,4.532779,11.686251
9,10.821197,4.145904,3.386363,14.798389


In [18]:
# 다중 대체 결과로부터 분산 계산
np.array(imputed_datasets).var(axis = 0)

array([[3.15544362e-30, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 9.22412247e-03, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [3.15544362e-30, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [3.69846499e-01, 7.88860905e-31, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 9.27273089e-02, 4.70095682e-03, 0.00000000e+00],
       [1.73092135e-01, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 2.73365386e-03, 0.00000000e+00]])